In [0]:

for folder in ['Customers', 'Branches', 'Transactions', 'Digital_Activity',
               'Loan_Payment', 'Customer_Transactions', 'Notifications',
               'Accounts', 'Cards', 'Loans', 'Merchants']:
    try:
        files = dbutils.fs.ls(f'/Volumes/customer_360/source_data/raw/{folder}/')
        print(f"\n{folder}/")
        for f in files:
            print(f"  {f.name}  ({f.size:,} bytes)")
    except:
        print(f"\n{folder}/ — folder not found")

In [0]:
%sql
SELECT
  CASE WHEN customer_id LIKE 'CUST%' THEN 'SYNTHETIC (CUST*)'
       ELSE 'REAL (alphanumeric)'
  END                    AS id_format,
  COUNT(*)               AS row_count,
  MIN(customer_id)       AS sample_min,
  MAX(customer_id)       AS sample_max
FROM customer_360.bronze.bronze_customers
GROUP BY CASE WHEN customer_id LIKE 'CUST%' THEN 'SYNTHETIC (CUST*)'
              ELSE 'REAL (alphanumeric)' END;

In [0]:
%sql
SELECT
  CASE WHEN branch_id LIKE 'BR%' THEN 'SYNTHETIC (BR*)'
       ELSE 'REAL (alphanumeric)'
  END                    AS id_format,
  COUNT(*)               AS row_count,
  MIN(branch_id)         AS sample_min,
  MAX(branch_id)         AS sample_max
FROM customer_360.bronze.bronze_branches
GROUP BY CASE WHEN branch_id LIKE 'BR%' THEN 'SYNTHETIC (BR*)'
              ELSE 'REAL (alphanumeric)' END;

In [0]:
for folder in ['Customers', 'Branches']:
    print(f"\n{folder}/ after cleanup:")
    for f in dbutils.fs.ls(f'/Volumes/customer_360/source_data/raw/{folder}/'):
        print(f"  {f.name}")

In [0]:
%sql
CREATE OR REPLACE TABLE customer_360.bronze.bronze_customers AS
SELECT
    *,
    current_timestamp() AS ingestion_time,
    _metadata.file_name AS source_file
FROM read_files(
    '/Volumes/customer_360/source_data/raw/Customers/',
    format => 'CSV',
    sep => ',',
    header => true,
    schema => '
       customer_id STRING,
       first_name  STRING,
       last_name   STRING,
       email       STRING,
       city        STRING,
       credit_score INTEGER,
       created_at  DATE
    ',
    rescuedDataColumn => '_rescued_data'
);

-- Verify: must show REAL (alphanumeric), row count must be 50,000
SELECT
  CASE WHEN customer_id LIKE 'CUST%' THEN 'SYNTHETIC — STILL WRONG'
       ELSE 'REAL — CORRECT'
  END      AS id_check,
  COUNT(*) AS rows
FROM customer_360.bronze.bronze_customers
GROUP BY 1;

In [0]:
%sql
SELECT
  LENGTH(customer_id)   AS id_length,
  LEFT(customer_id, 4)  AS prefix,
  COUNT(*)              AS count
FROM customer_360.bronze.bronze_customers
GROUP BY LENGTH(customer_id), LEFT(customer_id, 4)
ORDER BY id_length, prefix;

In [0]:

%sql
SELECT
  COUNT(CASE WHEN LENGTH(customer_id) = 8  THEN 1 END) AS actual_synthetic_remaining, -- must be 0
  COUNT(CASE WHEN LENGTH(customer_id) = 15 THEN 1 END) AS real_ids,                   -- must = total rows
  COUNT(*)                                              AS total_rows
FROM customer_360.silver.silver_digital_activity;

In [0]:
%sql
SELECT
  COUNT(CASE WHEN LENGTH(customer_id) = 8 THEN 1 END)  AS actual_synthetic_customer_ids, -- must be 0
  COUNT(CASE WHEN LENGTH(branch_id)   < 10 THEN 1 END) AS actual_synthetic_branch_ids,   -- must be 0
  COUNT(*)                                              AS total_rows
FROM customer_360.silver.silver_customer_interactions;